# Qwen3-VL to CLAP encoder-space projection

Run the cells in order. The initialization aligns bundles by prompt ID, uses `train` for fitting, `val` as the offline unseen test, and keeps the 50 `esc50_test` prompts target-free.

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter


def load_pack(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def find_uploaded_file(filename):
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path.cwd(),
        Path.home() / "Downloads",
    ]
    matches = []
    for root in search_roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    matches = list(dict.fromkeys(path.resolve() for path in matches))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {filename}; found {matches}. "
            "Set its path explicitly if more than one copy is uploaded."
        )
    return matches[0]


QWEN_PATH = find_uploaded_file("qwen3vl_encoder_all.pt")
CLAP_PATH = find_uploaded_file("clap_encoder_fit.pt")
print("Qwen:", QWEN_PATH)
print("CLAP:", CLAP_PATH)

qwen_pack = load_pack(QWEN_PATH)
clap_pack = load_pack(CLAP_PATH)

assert qwen_pack["encoder_features"].ndim == 2
assert clap_pack["encoder_features"].ndim == 2
assert qwen_pack["encoder_features"].shape[1] == 2048
assert clap_pack["encoder_features"].shape[1] == 768

qwen_index = {row_id: i for i, row_id in enumerate(qwen_pack["ids"])}
clap_index = {row_id: i for i, row_id in enumerate(clap_pack["ids"])}
assert len(qwen_index) == len(qwen_pack["ids"]), "Duplicate Qwen prompt ids"
assert len(clap_index) == len(clap_pack["ids"]), "Duplicate CLAP prompt ids"
assert set(clap_index).issubset(qwen_index), "CLAP fit ids must exist in Qwen bundle"

clap_rows = {row["id"]: row for row in clap_pack["rows"]}
qwen_rows = {row["id"]: row for row in qwen_pack["rows"]}

train_ids = [
    row_id for row_id in clap_pack["ids"]
    if clap_rows[row_id]["split"] == "train"
]
val_ids = [
    row_id for row_id in clap_pack["ids"]
    if clap_rows[row_id]["split"] == "val"
]
esc50_ids = [
    row_id for row_id in qwen_pack["ids"]
    if qwen_rows[row_id]["split"] == "esc50_test"
]


def aligned_features(pack, index, row_ids):
    return pack["encoder_features"][[index[row_id] for row_id in row_ids]].float()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Names expected by the existing nonlinear-kernel cell.
X = aligned_features(qwen_pack, qwen_index, train_ids).to(device)
Y = aligned_features(clap_pack, clap_index, train_ids).to(device)
X_unseen = aligned_features(qwen_pack, qwen_index, val_ids).to(device)
Y_unseen = aligned_features(clap_pack, clap_index, val_ids).to(device)

# True ESC-50 test inputs have no native CLAP targets and are never used for fitting.
X_esc50 = aligned_features(qwen_pack, qwen_index, esc50_ids).to(device)

class_names = [qwen_rows[row_id]["text"] for row_id in train_ids]
unseen_class_names = [qwen_rows[row_id]["text"] for row_id in val_ids]
esc50_texts = [qwen_rows[row_id]["text"] for row_id in esc50_ids]
esc50_class_names = [qwen_rows[row_id]["class_name"] for row_id in esc50_ids]
target_key = "encoder_features"

print("Qwen splits:", Counter(row["split"] for row in qwen_pack["rows"]))
print("CLAP splits:", Counter(row["split"] for row in clap_pack["rows"]))
print("train:      ", tuple(X.shape), "->", tuple(Y.shape))
print("validation: ", tuple(X_unseen.shape), "->", tuple(Y_unseen.shape))
print("ESC-50 test:", tuple(X_esc50.shape), "-> target intentionally absent")

assert X.shape == (5864, 2048)
assert Y.shape == (5864, 768)
assert X_unseen.shape == (198, 2048)
assert Y_unseen.shape == (198, 768)
assert X_esc50.shape == (50, 2048)


## Nonlinear kernel

The next cell is copied unchanged from `calculatingw (13).ipynb`. It evaluates on the 198 validation prompts.

In [ ]:
# ============================================================
# SAM-Audio: Polynomial Kernel Ridge Regression
# Cholesky is used only to solve the kernel system.
#
# Required tensors from the SAM-Audio initialization cell:
#   X          : seen source embeddings
#   Y          : seen SAM-Audio target embeddings
#   X_unseen   : unseen source embeddings
#   Y_unseen   : unseen SAM-Audio target embeddings
# ============================================================

import torch
import torch.nn.functional as F


# -------------------- Settings --------------------

SA_DEGREE = 2
SA_COEF0 = 1.0
SA_GAMMA = 1       # None means 1 / source_embedding_dimension
SA_LAMBDA = 1e-3
SA_SAVE_PATH = "Y_pred_unseen_runtime.pt"


# -------------------- Validation --------------------

required_names = ["X", "Y", "X_unseen", "Y_unseen"]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"{name} is missing. Run the SAM-Audio initialization cell first."
        )

if X.ndim != 2 or Y.ndim != 2:
    raise ValueError(
        f"X and Y must be matrices, but got X={X.shape}, Y={Y.shape}"
    )

if X_unseen.ndim != 2 or Y_unseen.ndim != 2:
    raise ValueError(
        "X_unseen and Y_unseen must be matrices, but got "
        f"X_unseen={X_unseen.shape}, Y_unseen={Y_unseen.shape}"
    )

if X.shape[0] != Y.shape[0]:
    raise ValueError(
        f"Seen row mismatch: X has {X.shape[0]} rows, "
        f"but Y has {Y.shape[0]} rows."
    )

if X_unseen.shape[0] != Y_unseen.shape[0]:
    raise ValueError(
        f"Unseen row mismatch: X_unseen has {X_unseen.shape[0]} rows, "
        f"but Y_unseen has {Y_unseen.shape[0]} rows."
    )

if X.shape[1] != X_unseen.shape[1]:
    raise ValueError(
        f"Source dimension mismatch: {X.shape[1]} vs {X_unseen.shape[1]}"
    )

if Y.shape[1] != Y_unseen.shape[1]:
    raise ValueError(
        f"Target dimension mismatch: {Y.shape[1]} vs {Y_unseen.shape[1]}"
    )


# -------------------- Prepare tensors --------------------

# Use the same device as X and float64 for a more stable Cholesky solve.
sa_device = X.device

sa_X_train = X.detach().to(device=sa_device, dtype=torch.float64)
sa_Y_train = Y.detach().to(device=sa_device, dtype=torch.float64)

sa_X_test = X_unseen.detach().to(device=sa_device, dtype=torch.float64)
sa_Y_test = Y_unseen.detach().to(device=sa_device, dtype=torch.float64)

# Normalize source and target embeddings row-wise.
# No centering or Cholesky whitening is applied to target embeddings.
sa_X_train = F.normalize(sa_X_train, p=2, dim=1)
sa_X_test = F.normalize(sa_X_test, p=2, dim=1)

sa_Y_train = F.normalize(sa_Y_train, p=2, dim=1)
sa_Y_test = F.normalize(sa_Y_test, p=2, dim=1)

sa_gamma = (
    1.0 / sa_X_train.shape[1]
    if SA_GAMMA is None
    else float(SA_GAMMA)
)


# -------------------- Polynomial kernel --------------------

def sa_polynomial_kernel(A, B):
    similarity = A @ B.T
    return (sa_gamma * similarity + SA_COEF0) ** SA_DEGREE


sa_K_train = sa_polynomial_kernel(sa_X_train, sa_X_train)
sa_K_unseen = sa_polynomial_kernel(sa_X_test, sa_X_train)

# Remove small numerical asymmetry before Cholesky.
sa_K_train = 0.5 * (sa_K_train + sa_K_train.T)


# -------------------- Cholesky KRR solve --------------------

sa_identity = torch.eye(
    sa_K_train.shape[0],
    device=sa_device,
    dtype=sa_K_train.dtype,
)

# Increase regularization automatically only if Cholesky initially fails.
sa_effective_lambda = float(SA_LAMBDA)

for sa_attempt in range(8):
    sa_system = sa_K_train + sa_effective_lambda * sa_identity
    sa_L, sa_info = torch.linalg.cholesky_ex(sa_system)

    if sa_info.max().item() == 0:
        break

    sa_effective_lambda *= 10.0
else:
    raise RuntimeError(
        "Cholesky decomposition failed even after increasing regularization."
    )

# Solve (K + lambda*I) @ alpha = Y
sa_alpha = torch.cholesky_solve(sa_Y_train, sa_L)


# -------------------- Predictions --------------------

# Predict first, then normalize the final target embeddings.
sa_Y_pred_seen = F.normalize(
    sa_K_train @ sa_alpha,
    p=2,
    dim=1,
)

sa_Y_pred_unseen = F.normalize(
    sa_K_unseen @ sa_alpha,
    p=2,
    dim=1,
)


# -------------------- Evaluation --------------------

def sa_projection_report(prediction, target, split_name):
    prediction = F.normalize(prediction, p=2, dim=1)
    target = F.normalize(target, p=2, dim=1)

    similarity = prediction @ target.T
    correct_similarity = similarity.diag()

    non_diagonal_mask = ~torch.eye(
        similarity.shape[0],
        dtype=torch.bool,
        device=similarity.device,
    )
    wrong_similarity = similarity[non_diagonal_mask]

    predicted_indices = similarity.argmax(dim=1)
    correct_indices = torch.arange(
        similarity.shape[0],
        device=similarity.device,
    )
    top1_accuracy = (
        predicted_indices == correct_indices
    ).double().mean()

    cosine_per_pair = (prediction * target).sum(dim=1)
    mse = F.mse_loss(prediction, target)
    l2_per_pair = torch.linalg.vector_norm(
        prediction - target,
        dim=1,
    )

    print(f"\n--- {split_name} ---")
    print(
        f"Correct cosine mean: {correct_similarity.mean().item():.6f}"
    )
    print(
        f"Wrong cosine mean:   {wrong_similarity.mean().item():.6f}"
    )
    print(
        "Separation gap:      "
        f"{(correct_similarity.mean() - wrong_similarity.mean()).item():.6f}"
    )
    print(f"Top-1 accuracy:      {top1_accuracy.item():.6f}")
    print(f"Paired cosine mean:  {cosine_per_pair.mean().item():.6f}")
    print(f"Normalized MSE:      {mse.item():.8f}")
    print(f"Mean normalized L2:  {l2_per_pair.mean().item():.6f}")


print("SAM-Audio polynomial KRR")
print(f"Train shapes:  X={tuple(sa_X_train.shape)}, Y={tuple(sa_Y_train.shape)}")
print(f"Unseen shapes: X={tuple(sa_X_test.shape)}, Y={tuple(sa_Y_test.shape)}")
print(f"Degree:        {SA_DEGREE}")
print(f"Gamma:         {sa_gamma}")
print(f"Coef0:         {SA_COEF0}")
print(f"Requested λ:   {SA_LAMBDA}")
print(f"Effective λ:   {sa_effective_lambda}")

sa_projection_report(
    sa_Y_pred_seen,
    sa_Y_train,
    "seen/runtime space",
)

sa_projection_report(
    sa_Y_pred_unseen,
    sa_Y_test,
    "unseen/runtime space",
)


# -------------------- Save runtime-space output --------------------

# Save using the original target dtype and on CPU.
Y_pred = sa_Y_pred_seen.to(
    device="cpu",
    dtype=Y.dtype,
)

Y_pred_unseen = sa_Y_pred_unseen.to(
    device="cpu",
    dtype=Y_unseen.dtype,
)

torch.save(Y_pred_unseen, SA_SAVE_PATH)

print(f"\nSaved unseen runtime embeddings to: {SA_SAVE_PATH}")
print(f"Saved tensor shape: {tuple(Y_pred_unseen.shape)}")

## Produce ESC-50 evaluation features

This applies the fitted kernel to the 50 held-out Qwen features and saves the `[50,768]` bundle expected by the CLAP projected evaluator.

In [ ]:
# Apply the fitted kernel to the 50 target-free ESC-50 Qwen features.
# Run this only after the unchanged nonlinear-kernel cell above finishes.
required = ["sa_X_train", "sa_alpha", "sa_L", "sa_polynomial_kernel"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Run the nonlinear-kernel cell first; missing {missing}")

sa_X_esc50 = F.normalize(
    X_esc50.detach().to(device=sa_device, dtype=torch.float64),
    p=2,
    dim=1,
)
sa_K_esc50 = sa_polynomial_kernel(sa_X_esc50, sa_X_train)

# The unchanged KRR cell predicts target directions in normalized CLAP space.
sa_esc50_direction = F.normalize(sa_K_esc50 @ sa_alpha, p=2, dim=1)

# CLAP's frozen 768->1024 projection is scale-sensitive. Predict the raw GPT-2
# feature norm with the same kernel system, then restore it before evaluation.
sa_train_target_norms = torch.linalg.vector_norm(
    Y.detach().to(device=sa_device, dtype=torch.float64),
    dim=1,
    keepdim=True,
)
sa_norm_alpha = torch.cholesky_solve(sa_train_target_norms, sa_L)
sa_esc50_norms = sa_K_esc50 @ sa_norm_alpha
sa_esc50_norms = sa_esc50_norms.clamp(
    min=sa_train_target_norms.min(),
    max=sa_train_target_norms.max(),
)

projected_esc50 = (sa_esc50_direction * sa_esc50_norms).float().cpu()
assert projected_esc50.shape == (50, 768)
assert torch.isfinite(projected_esc50).all()

OUTPUT_PATH = Path("/kaggle/working/projected_esc50_encoder_features.pt")
if not Path("/kaggle/working").exists():
    OUTPUT_PATH = Path.cwd() / "projected_esc50_encoder_features.pt"

torch.save(
    {
        "kind": "projected_clap_encoder_features",
        "source_model": qwen_pack["model_id"],
        "target_model": clap_pack["model_id"],
        "ids": esc50_ids,
        "texts": esc50_texts,
        "class_names": esc50_class_names,
        "projected_encoder_features": projected_esc50,
        "hidden_dim": 768,
        "kernel": {
            "name": "polynomial",
            "degree": SA_DEGREE,
            "gamma": sa_gamma,
            "coef0": SA_COEF0,
            "requested_lambda": SA_LAMBDA,
            "effective_lambda": sa_effective_lambda,
        },
        "norm_restoration": "polynomial KRR using raw CLAP train norms",
    },
    OUTPUT_PATH,
)

print("Saved:", OUTPUT_PATH)
print("Projected ESC-50 shape:", tuple(projected_esc50.shape))
print(
    "Projected norm mean/std:",
    projected_esc50.norm(dim=1).mean().item(),
    projected_esc50.norm(dim=1).std().item(),
)
print("CLAP train norm mean/std:", Y.norm(dim=1).mean().item(), Y.norm(dim=1).std().item())
